# Data Preprocessing Notebook - Global Education Statistics Dashboard
## Project: Data Visualization Final Project
### Dataset: World Education Data

This notebook performs all data cleaning, transformation, and preparation steps for the Global Education Statistics Dashboard.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from datetime import datetime

print("Libraries imported successfully!")

## Step 1: Load the Dataset

In [ ]:
# Load the dataset
# Note: The dataset will be downloaded from Kaggle or provided locally
# For this project, we'll create a comprehensive education dataset based on the World Education Data structure

# Since we need to work with the actual dataset structure, let's create a representative dataset
# based on the World Education Data from Kaggle

df = pd.read_csv('Global_Education.csv')
print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape}")
df.head()

## Step 2: Initial Data Exploration

In [ ]:
# Display basic information about the dataset
print("=" * 50)
print("DATASET INFO")
print("=" * 50)
print(df.info())
print("\n")

print("=" * 50)
print("DESCRIPTIVE STATISTICS")
print("=" * 50)
print(df.describe())
print("\n")

print("=" * 50)
print("COLUMN NAMES")
print("=" * 50)
print(df.columns.tolist())

## Step 3: Handle Missing Values

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_percentage
})

print("Missing Values Summary:")
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Handle missing values strategically
# For numerical columns, we'll use median imputation for critical metrics
# For categorical columns, we'll use mode or create an 'Unknown' category

# Create a copy of the dataframe for cleaning
df_clean = df.copy()

# Identify key columns for our analysis
key_columns = [
    'Country', 
    'Region', 
    'Literacy rate', 
    'GDP per capita', 
    'Government spending on education (% of GDP)',
    'Student-teacher ratio',
    'School life expectancy',
    'Enrollment rates'
]

# Check which columns exist in our dataset
existing_columns = [col for col in key_columns if col in df_clean.columns]
print(f"Available key columns: {existing_columns}")

# Fill missing values for numerical columns with median
numerical_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    if df_clean[col].isnull().any():
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        print(f"Filled missing values in '{col}' with median: {median_val}")

# Fill missing values for categorical columns with mode
categorical_cols = df_clean.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df_clean[col].isnull().any():
        mode_val = df_clean[col].mode()[0]
        df_clean[col] = df_clean[col].fillna(mode_val)
        print(f"Filled missing values in '{col}' with mode: {mode_val}")

print("\nMissing values handled successfully!")

## Step 4: Data Transformation

In [ ]:
# Rename columns for consistency and easier handling
column_mapping = {}
for col in df_clean.columns:
    # Convert to lowercase and replace spaces with underscores
    new_name = col.lower().replace(' ', '_').replace('-', '_').replace('(', '').replace(')', '')
    column_mapping[col] = new_name

df_clean = df_clean.rename(columns=column_mapping)
print("Column renaming mapping:")
print(column_mapping)
print(f"\nNew column names: {df_clean.columns.tolist()}")

In [ ]:
# Ensure proper data types
# Convert year columns to numeric if they exist
year_columns = [col for col in df_clean.columns if 'year' in col.lower() or col.isdigit()]
for col in year_columns:
    try:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        print(f"Converted '{col}' to numeric")
    except:
        pass

# Ensure country and region are strings
if 'country' in df_clean.columns:
    df_clean['country'] = df_clean['country'].astype(str)
if 'region' in df_clean.columns:
    df_clean['region'] = df_clean['region'].astype(str)

print("\nData types after conversion:")
print(df_clean.dtypes)

## Step 5: Feature Engineering

In [ ]:
# Create additional features for analysis

# Add continent/region categorization if not present
if 'region' in df_clean.columns:
    # Create broader region categories
    def categorize_region(region):
        region_lower = str(region).lower()
        if any(x in region_lower for x in ['africa', 'sub-saharan']):
            return 'Africa'
        elif any(x in region_lower for x in ['asia', 'eastern asia', 'southern asia', 'south-eastern asia']):
            return 'Asia'
        elif any(x in region_lower for x in ['europe', 'eastern europe', 'western europe']):
            return 'Europe'
        elif any(x in region_lower for x in ['america', 'latin america', 'caribbean', 'north america']):
            return 'Americas'
        elif any(x in region_lower for x in ['oceania', 'pacific']):
            return 'Oceania'
        elif any(x in region_lower for x in ['middle east', 'arab']):
            return 'Middle East'
        else:
            return 'Other'
    
    df_clean['continent'] = df_clean['region'].apply(categorize_region)
    print("Created 'continent' feature based on region")

# Create education performance categories
if 'literacy_rate' in df_clean.columns:
    def categorize_literacy(rate):
        if rate >= 95:
            return 'Very High'
        elif rate >= 85:
            return 'High'
        elif rate >= 70:
            return 'Medium'
        else:
            return 'Low'
    
    df_clean['literacy_category'] = df_clean['literacy_rate'].apply(categorize_literacy)
    print("Created 'literacy_category' feature")

# Create GDP per capita categories
if 'gdp_per_capita' in df_clean.columns:
    median_gdp = df_clean['gdp_per_capita'].median()
    df_clean['income_level'] = df_clean['gdp_per_capita'].apply(
        lambda x: 'High Income' if x > median_gdp * 1.5 else ('Middle Income' if x > median_gdp * 0.5 else 'Low Income')
    )
    print("Created 'income_level' feature")

print("\nFeature engineering completed!")
print(f"Final dataset shape: {df_clean.shape}")

## Step 6: Data Validation

In [ ]:
# Validate the cleaned data
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)

# Check for duplicates
duplicates = df_clean.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

# Remove duplicates if any
if duplicates > 0:
    df_clean = df_clean.drop_duplicates()
    print(f"Duplicates removed. New shape: {df_clean.shape}")

# Check for outliers in key numerical columns
print("\nOutlier Detection (using IQR method):")
numerical_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in numerical_cols[:5]:  # Check first 5 numerical columns
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df_clean[col] < Q1 - 1.5 * IQR) | (df_clean[col] > Q3 + 1.5 * IQR)).sum()
    print(f"  {col}: {outliers} outliers detected")

print("\nData validation completed!")

## Step 7: Save Cleaned Dataset

In [ ]:
# Save the cleaned dataset
output_file = 'cleaned_education_data.csv'
df_clean.to_csv(output_file, index=False)
print(f"Cleaned dataset saved to '{output_file}'")
print(f"Final dataset shape: {df_clean.shape}")
print(f"\nColumns in final dataset: {df_clean.columns.tolist()}")

# Display sample of cleaned data
print("\nSample of cleaned data:")
display(df_clean.head())

## Step 8: Summary Statistics for Dashboard Planning

In [ ]:
# Generate summary statistics to help plan dashboard visualizations
print("=" * 50)
print("SUMMARY STATISTICS FOR DASHBOARD PLANNING")
print("=" * 50)

# Unique values in categorical columns
categorical_cols = df_clean.select_dtypes(include=['object']).columns
for col in categorical_cols:
    unique_count = df_clean[col].nunique()
    print(f"\n{col}:")
    print(f"  Unique values: {unique_count}")
    if unique_count <= 20:
        print(f"  Values: {df_clean[col].unique()[:10].tolist()}")

# Key metrics statistics
print("\n" + "=" * 50)
print("KEY METRICS SUMMARY")
print("=" * 50)
key_metrics = ['literacy_rate', 'gdp_per_capita', 'government_spending_on_education', 
               'student_teacher_ratio', 'school_life_expectancy']
for metric in key_metrics:
    if metric in df_clean.columns:
        print(f"\n{metric.replace('_', ' ').title()}:")
        print(f"  Min: {df_clean[metric].min():.2f}")
        print(f"  Max: {df_clean[metric].max():.2f}")
        print(f"  Mean: {df_clean[metric].mean():.2f}")
        print(f"  Median: {df_clean[metric].median():.2f}")
        print(f"  Std Dev: {df_clean[metric].std():.2f}")

print("\n" + "=" * 50)
print("DATA PREPROCESSING COMPLETED SUCCESSFULLY!")
print("=" * 50)